In [1]:
import pandas as pd
import pickle
import numpy as np
import time

# Data Analysis

This dataset simulates an MQTT-based (Message Queuing Telemetry Transport) network. The network comprises twelve sensors, a broker, a simulated camera, and an attacker. Five scenarios are recorded:

1. normal operation
2. aggressive scan
3. UDP scan
4. Sparta SSH brute-force
5. MQTT brute-force attack

The raw PCAP files are saved, then features are extracted. Three abstraction levels of features are extracted from the raw PCAP files:
- packet features
- unidirectional flow features
- bidirectional flow features

The CSV feature files in the dataset are suited for Machine Learning (ML) usage. Also, the raw PCAP files are suitable for the deeper analysis of MQTT IoT networks communication and the associated attacks.

The dataset consists of 5 PCAP files representing a recording of one scenario:
- `normal.pcap`: normal operation
- `sparta.pcap`: Sparta SSH brute-force
- `scan_A.pcap`: aggressive scan
- `mqtt_bruteforce.pcap`: MQTT brute-force
- `scan_sU.pcap`: UDP scan

The attacker IP address is `192.168.2.5`. Basic packet features are extracted from the pcap files into CSV files with the same pcap file names. The features include flags, length, MQTT message parameters, etc. Later, unidirectional and bidirectional features are extracted. It is important to note that for the bidirectional flows, some features (pointed as *) have two values-one for forward flow and one for the backward flow. The two features are recorded and distinguished by a prefix “fwd” for forward and “bwd” for backward.

## Packet features

IP fields and flags (<a href="https://www.eit.lth.se/ppplab/IPHeader.htm">IPv4 packet structure</a>):
- `timestamp`: date and sending time of the packet
- `src_ip`: IP address of the source
- `dst_ip`: IP address of the destination
- `protocol`: this field specifies the next encapsulated protocol, one of `DATA`, `DNS`, `DVB_SDT`, `MDNS`, `MP2T`, `MPEG_PAT`, `MPEG_PMT`, `MQTT`, `TCP`
- `ttl`: time-to-live of the packet, if 0 then the datagram should be discarded
- `ip_len`: the number of 32 bit-segments in the IP header
- `ip_flag_df`: do not fragment. Controls the fragmentation of the datagram
- `ip_flag_mf`: more fragments. Indicates if the datagram contains additional fragments
- `ip_flag_rb`:

TCP fields and flags (<a href="https://en.wikipedia.org/wiki/Transmission_Control_Protocol">TCP-specific flags</a>):
- `src_port`: source port
- `dst_port`: destination port
- `tcp_flag_res`: reserved bits, should be set to 0
- `tcp_flag_ns`: ECN (Explicit Congestion Notification) nonce - concealment protection. An optional addition to ECN that protects against accidental or malicious concealment of marked packets from the TCP sender. It improves the robustness of congestion control by preventing receivers from exploiting ECN to gain an unfair share of network bandwidth. The ECN-nonce uses the two ECN-Capable Transport (ECT)codepoints in the ECN field of the IP header, and requires a flag in the TCP header. It is computationally efficient for both routers and hosts. <a href="https://tools.ietf.org/html/rfc3540">Source</a>
- `tcp_flag_cwr`: congestion window reduced (CWR) flag is set by the sending host to indicate that it received a TCP segment with the ECE flag set and had responded in congestion control mechanism
- `tcp_flag_ecn`: 
- `tcp_flag_urg`: indicates that the Urgent pointer field is significant
- `tcp_flag_ack`: indicates that the Acknowledgment field is significant. All packets after the initial SYN packet sent by the client should have this flag set
- `tcp_flag_push`: push function. Asks to push the buffered data to the receiving application
- `tcp_flag_reset`: reset the connection
- `tcp_flag_syn`: synchronize sequence numbers. Only the first packet sent from each end should have this flag set. Some other flags and fields change meaning based on this flag, and some are only valid when it is set, and others when it is clear
- `tcp_flag_fin`: set to 1 if last packet from the sender

MQTT protocol fields and flags (<a href="https://docs.solace.com/MQTT-311-Prtl-Conformance-Spec/MQTT%20Control%20Packet%20format.htm">MQTT packet</a>):
- `mqtt_messagetype`:
- `mqtt_messagelength`: number of bytes remaining within the current packet, including data in the variable header and the payload. The remaining length does not include the bytes used to encode the remaining length
- `mqtt_flag_uname`: MQTT auth flag
- `mqtt_flag_passwd`: MQTT auth flag
- `mqtt_flag_retain`: a retained message is a normal MQTT message. The broker stores the last retained message and the corresponding QoS for that topic. Each client that subscribes to a topic pattern that matches the topic of the retained message receives the retained message immediately after they subscribe. The broker stores only one retained message per topic
- `mqtt_flag_qos`: quality of service flag. '0': at most once - for no guaranteed delivery, '1': at least once - the message can be guaranteed to be published at least once, and '2': exactly once - publishers and subscribers ensure that messages are published only once through two sessions. This is the highest level of service quality, and message loss and duplication are unacceptable
- `mqtt_flag_willflag`: when the client disconnects, a will message is sent to the relevant subscriber
- `mqtt_flag_clean`: indicate whether or not a persistent connection is required
- `mqtt_flag_reserved`: reserved bits, should be set to zero

Finally, the labels that indicate if that packet belongs to an attack
- `is_attack`: 0 or 1

In [5]:
t_start = time.time()

# 1.112 GB (approx. 3min loading time)
#mqtt_bruteforce = pd.read_csv('./data/packet_features/mqtt_bruteforce.csv')

# 0.114 GB
normal = pd.read_csv('./data/packet_features/normal.csv')

# 0.012 GB
#scan_a = pd.read_csv('./data/packet_features/scan_A.csv')

# 0.024 GB
#scan_su = pd.read_csv('./data/packet_features/scan_sU.csv')

# 2.171 GB (approx. 7 min loading time)
#sparta = pd.read_csv('./data/packet_features/sparta.csv')

t_finish = time.time()
print(f"Dataset loaded in {round((t_finish - t_start) / 60, 4)} min.")

Dataset loaded in 0.0795 min.


In [6]:
normal.head()

,timestamp,src_ip,dst_ip,protocol,ttl,ip_len,ip_flag_df,ip_flag_mf,ip_flag_rb,src_port,...,mqtt_messagetype,mqtt_messagelength,mqtt_flag_uname,mqtt_flag_passwd,mqtt_flag_retain,mqtt_flag_qos,mqtt_flag_willflag,mqtt_flag_clean,mqtt_flag_reserved,is_attack
0,"02/14/2020, 10:09:22:951038",10.0.0.5,192.168.1.7,TCP,64,60,1,0,0,56572,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
1,"02/14/2020, 10:09:22:951481",192.168.1.7,10.0.0.5,TCP,63,60,1,0,0,1883,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
2,"02/14/2020, 10:09:22:951706",10.0.0.5,192.168.1.7,TCP,64,52,1,0,0,56572,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
3,"02/14/2020, 10:09:22:951787",10.0.0.5,192.168.1.7,MQTT,64,101,1,0,0,56572,...,1,47,1,1,0,0,0,1,0,0
4,"02/14/2020, 10:09:22:951913",192.168.1.7,10.0.0.5,TCP,63,52,1,0,0,1883,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0


In [7]:
#len(normal)
#len(mqtt_bruteforce)
len(sparta)

20676140

## Uniflow features

In [5]:
#mqtt_bruteforce = pd.read_csv('./data/uniflow_features/uniflow_mqtt_bruteforce.csv')
normal = pd.read_csv('./data/uniflow_features/uniflow_normal.csv')
#scan_a = pd.read_csv('./data/uniflow_features/uniflow_scan_A.csv')
#scan_sU = pd.read_csv('./data/uniflow_features/uniflow_scan_sU.csv')
#sparta = pd.read_csv('./data/uniflow_features/uniflow_sparta.csv')

In [6]:
normal.head()

,ip_src,ip_dst,prt_src,prt_dst,proto,num_pkts,mean_iat,std_iat,min_iat,max_iat,mean_pkt_len,num_bytes,num_psh_flags,num_rst_flags,num_urg_flags,std_pkt_len,min_pkt_len,max_pkt_len,is_attack
0,10.0.0.5,192.168.1.7,56572,1883,6,7,0.000237,0.000218,0.000035,0.000668,63.428571,444,3,0,0,16.884965,52.0,101.0,0
1,192.168.1.7,10.0.0.5,1883,56572,6,5,0.000203,0.000193,0.000004,0.000432,54.400000,272,1,0,0,3.200000,52.0,60.0,0
2,10.0.0.23,192.168.2.7,47103,1234,17,73552,0.090855,0.060826,0.000002,0.165978,1344.000000,98853888,0,0,0,0.000000,1344.0,1344.0,0
3,10.0.0.11,192.168.1.7,51906,1883,6,7,0.000242,0.000207,0.000013,0.000555,69.000000,483,3,0,0,24.041631,52.0,112.0,0
4,192.168.1.7,10.0.0.11,1883,51906,6,5,0.000243,0.000231,0.000004,0.000484,54.400000,272,1,0,0,3.200000,52.0,60.0,0


In [7]:
len(normal)

171836

## Biflow features

In [8]:
#mqtt_bruteforce = pd.read_csv('./data/biflow_features/biflow_mqtt_bruteforce.csv')
normal = pd.read_csv('./data/biflow_features/biflow_normal.csv')
#scan_a = pd.read_csv('./data/biflow_features/biflow_scan_A.csv')
#scan_sU = pd.read_csv('./data/biflow_features/biflow_scan_sU.csv')
#sparta = pd.read_csv('./data/biflow_features/biflow_sparta.csv')

In [9]:
normal.head()

,ip_src,ip_dst,prt_src,prt_dst,proto,fwd_num_pkts,bwd_num_pkts,fwd_mean_iat,bwd_mean_iat,fwd_std_iat,...,bwd_max_pkt_len,fwd_num_bytes,bwd_num_bytes,fwd_num_psh_flags,bwd_num_psh_flags,fwd_num_rst_flags,bwd_num_rst_flags,fwd_num_urg_flags,bwd_num_urg_flags,is_attack
0,10.0.0.5,192.168.1.7,56572,1883,6,7,5,0.000237,0.000203,0.000218,...,60.0,444,272,3,1,0,0,0,0,0
1,10.0.0.11,192.168.1.7,51906,1883,6,7,5,0.000242,0.000243,0.000207,...,60.0,483,272,3,1,0,0,0,0,0
2,10.0.0.14,192.168.1.7,54202,1883,6,7,5,0.000284,0.000349,0.000323,...,60.0,531,272,3,1,0,0,0,0,0
3,10.0.0.12,192.168.1.7,39782,1883,6,7,5,0.001038,0.000968,0.000895,...,60.0,462,272,3,1,0,0,0,0,0
4,10.0.0.13,192.168.1.7,38052,1883,6,7,5,0.001039,0.000968,0.000776,...,60.0,463,272,3,1,0,0,0,0,0


In [10]:
len(normal)

86008

In [ ]:
normal.isnull().sum()